# Connect Python to MySQL

In [2]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sqlalchemy import create_engine, URL, inspect
from getpass import getpass

In [11]:
username = "root"
password = getpass("Enter MySQL password: ")

database = "sample_sales"

connection_url = URL.create(
    "mysql+pymysql",
    username=username,
    password=password,
    host="127.0.0.1",
    port=3306,
    database=database
)

engine = create_engine(connection_url)

with engine.connect() as connection:
    print("Successfully connected!")

Enter MySQL password:  ········


Successfully connected!


In [12]:
inspector = inspect(engine)

tables = inspector.get_table_names()

print("Tables in database:")

for table in tables:
    print(table)

Tables in database:
Inventory_Categories
Inventory_Subcategories
Management
Online_Sales
Products
Shipper_List
Store_Locations
Store_Sales


In [13]:
for table in tables:
    print(f"\nTable: {table}")
    print("-" * 40)

    columns = inspector.get_columns(table)

    for column in columns:
        print(f"{column['name']} : {column['type']}")


Table: Inventory_Categories
----------------------------------------
Categoryid : INTEGER
Category : VARCHAR(255)
Description : VARCHAR(255)

Table: Inventory_Subcategories
----------------------------------------
Subcategoryid : VARCHAR(255)
Categoryid : INTEGER
Subcategory : VARCHAR(255)

Table: Management
----------------------------------------
id : INTEGER
Region : VARCHAR(255)
RegionalDirector : VARCHAR(255)
State : VARCHAR(255)
SalesManager : VARCHAR(255)

Table: Online_Sales
----------------------------------------
id : INTEGER
Date : DATE
ProdNum : VARCHAR(255)
SalesTotal : DECIMAL(10, 2)
ShiptoState : VARCHAR(255)

Table: Products
----------------------------------------
ProdNum : VARCHAR(255)
Product : VARCHAR(255)
Categoryid : INTEGER
Subcategoryid : VARCHAR(255)

Table: Shipper_List
----------------------------------------
id : INTEGER
ShiptoState : VARCHAR(255)
PreferredShipper : VARCHAR(255)
VolumeDiscount : DECIMAL(10, 2)

Table: Store_Locations
-----------------------

# Loading Data

In [15]:
import pandas as pd

# Load all database tables into pandas
dataframes = {}

for table in tables:
    dataframes[table] = pd.read_sql(
        f"SELECT * FROM `{table}`",
        engine
    )

print("All tables loaded successfully!")

All tables loaded successfully!


In [16]:
store_sales = dataframes["Store_Sales"]
online_sales = dataframes["Online_Sales"]
products = dataframes["Products"]
categories = dataframes["Inventory_Categories"]
subcategories = dataframes["Inventory_Subcategories"]
management = dataframes["Management"]
store_locations = dataframes["Store_Locations"]
shipper_list = dataframes["Shipper_List"]

In [18]:
# Understanding the table sizes.
for name, df in dataframes.items():
    print(f"{name}: {df.shape[0]} rows, {df.shape[1]} columns")

Inventory_Categories: 6 rows, 3 columns
Inventory_Subcategories: 52 rows, 3 columns
Management: 12 rows, 5 columns
Online_Sales: 125565 rows, 5 columns
Products: 669 rows, 4 columns
Shipper_List: 29 rows, 4 columns
Store_Locations: 111 rows, 3 columns
Store_Sales: 335129 rows, 5 columns


### Investigate the sales data

In [19]:
store_sales.head()

,id,Transaction_Date,Store_ID,Prod_Num,Sale_Amount
0,1,2022-01-01,702,105349-M,8.00
1,2,2022-01-01,704,105350-T,144.00
2,3,2022-01-01,705,105351-M,44.00
3,4,2022-01-01,705,105352-M,47.61
4,5,2022-01-01,705,105353-A,20.36


In [29]:
store_sales.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 335129 entries, 0 to 335128
Data columns (total 5 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   id                335129 non-null  int64  
 1   Transaction_Date  335129 non-null  object 
 2   Store_ID          335129 non-null  int64  
 3   Prod_Num          335129 non-null  object 
 4   Sale_Amount       335129 non-null  float64
dtypes: float64(1), int64(2), object(2)
memory usage: 12.8+ MB


In [31]:
store_sales.describe().T

,count,mean,std,min,25%,50%,75%,max
id,335129.0,167565.000000,96743.553519,1.0,83783.00,167565.00,251347.00,335129.0
Store_ID,335129.0,785.001125,63.119788,701.0,733.00,801.00,837.00,911.0
Sale_Amount,335129.0,135.380850,279.716725,2.4,13.13,29.81,114.71,1763.7


In [32]:
store_sales.isnull().sum()

id                  0
Transaction_Date    0
Store_ID            0
Prod_Num            0
Sale_Amount         0
dtype: int64

In [25]:
store_sales.duplicated().sum()

0

In [26]:
store_sales.shape

(335129, 5)